In [1]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [2]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [3]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(24)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-20 16:09:49, wtch_dt_end:2026-07-21 16:09:49


In [4]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [5]:
@file:DependsOn("org.json:json:20250107")

In [6]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [7]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2143,1079,0,0.260000,61,4.991109,2.130926,0.250000,3.780833,5.460000,6.370000,15.784000
rtmWqChpla,Comparable<*>,2143,1324,0,,178,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2143,1,0,,2143,null,null,,,,,
rtmWqWtchStaCd,String,2143,14,0,SEA1005,178,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2143,2143,0,1,1,1072.000000,618.775134,1,536.166667,1072.000000,1607.833333,2143
rtmWqTu,Int,2143,152,0,5,191,24.587961,31.989430,0,5.000000,11.000000,32.000000,232
ph,Double,2143,128,0,7.500000,53,7.668875,0.266566,7.020000,7.490000,7.640000,7.910000,8.760000
rtmWqSlnty,Number,2143,1984,0,32.737999,4,21.798058,10.324050,0.270000,13.946000,26.635000,29.556999,34.032001
rtmWqCndctv,Float,2143,2063,0,37.926998,3,34.038917,15.469434,0.558000,23.284166,39.754002,45.115168,54.451000
rtmWqWtchDtlDt,String,2143,190,0,2026-07-20 16:25:00.0,14,null,null,2026-07-20 16:10:00.0,2026-07-20 22:00:00.0,2026-07-21 03:55:00.0,2026-07-21 09:55:00.0,2026-07-21 15:45:00.0


In [8]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0.0 else value.toDouble()
}.convert {  colsOf<Number>() }.with { it.toString().trim().toDouble()}


df.describe()

kotlin-logging: initializing... active logger factory: Slf4jLoggerFactory


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2143,1079,0,0.260000,61,4.991109,2.130926,0.250000,3.780833,5.460000,6.370000,15.784000
rtmWqChpla,Double,2143,1324,0,0.000000,178,5.157429,5.228613,0.000000,1.300333,2.964000,7.910000,32.040000
rtmWqWtchStaCd,String,2143,14,0,SEA1005,178,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Double,2143,2143,0,1.000000,1,1072.000000,618.775134,1.000000,536.166667,1072.000000,1607.833333,2143.000000
rtmWqTu,Double,2143,152,0,5.000000,191,24.587961,31.989430,0.000000,5.000000,11.000000,32.000000,232.000000
ph,Double,2143,128,0,7.500000,53,7.668875,0.266566,7.020000,7.490000,7.640000,7.910000,8.760000
rtmWqSlnty,Double,2143,1984,0,32.737999,4,21.798058,10.324050,0.270000,13.948167,26.635000,29.557834,34.032001
rtmWqCndctv,Double,2143,2063,0,37.927000,3,34.038917,15.469434,0.558000,23.284167,39.754000,45.115167,54.451000
rtmWqWtchDtlDt,LocalDateTime,2143,190,0,2026-07-20T16:25,14,null,null,2026-07-20T16:10,2026-07-20T22:00,2026-07-21T03:55,2026-07-21T09:55,2026-07-21T15:45
rtmWtchWtem,Double,2143,757,0,26.990000,14,26.397783,2.340389,19.740000,24.991666,26.459999,28.190001,30.889999


In [9]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")

In [14]:
val renamedDf = df.rename(
    "rtmWqDoxn" to "용존산소",
    "rtmWqChpla" to "클로로필",
    "rtmWqWtchStaCd" to "관측정점코드",
    "num" to "순번",
    "rtmWqTu" to "탁도",
    "ph" to "수소이온농도",
    "rtmWqSlnty" to "염분",
    "rtmWqCndctv" to "전기전도도",
    "rtmWqWtchDtlDt" to "일시",
    "rtmWtchWtem" to "수온"
)

renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [11]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1.000000,2026-07-20T16:10,3.554000,4.337000,SEA2007,33.000000,7.920000,32.560001,52.471000,24.559999
2.000000,2026-07-20T16:10,6.081000,9.304000,SEA2005,2.000000,7.790000,28.176001,43.721000,30.660000
3.000000,2026-07-20T16:10,6.300000,2.770000,NEP2002,7.000000,7.970000,27.941999,42.791000,24.290001
4.000000,2026-07-20T16:10,0.950000,0.000000,SEA1005,11.000000,7.340000,2.544000,4.777000,27.750000
5.000000,2026-07-20T16:10,6.991000,3.698000,NEP2001,7.000000,7.910000,11.035000,18.638000,28.020000


In [12]:
removedDf
    .select{  일시 and 수온 and 관측정점코드   }
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(수온) {axis.name ="수온 °C"}
        line{
            color(관측정점코드){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점코드"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="5dEQfK" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("5dEQfK");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"일시":[1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7